# Lecture 10: Deep Learning for Text Classification — Answer Key
### NLP Course 2027 — Week 05

---
Complete answers for all four exercises in Lecture 10.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import nltk
from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

nltk.download('movie_reviews', quiet=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
# Shared data loading and vocab building (used by all exercises)
def load_movie_reviews(max_len=200, min_freq=2):
    docs, labels = [], []
    for cat, label in [('pos', 1), ('neg', 0)]:
        for fid in movie_reviews.fileids(cat):
            words = [w.lower() for w in movie_reviews.words(fid)]
            docs.append(words)
            labels.append(label)
    # Build vocab
    freq = Counter(w for doc in docs for w in doc)
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for w, c in freq.items():
        if c >= min_freq:
            vocab[w] = len(vocab)
    # Encode and pad
    encoded = [[vocab.get(w, 1) for w in doc[:max_len]] for doc in docs]
    padded  = [seq + [0] * (max_len - len(seq)) for seq in encoded]
    return padded, labels, vocab

class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

X, y, vocab = load_movie_reviews()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader  = DataLoader(TextDataset(X_test,  y_test),  batch_size=64)
print(f'Vocab size: {len(vocab):,} | Train: {len(X_train)} | Test: {len(X_test)}')

---
## Exercise 10.1 — TextCNN

**Task:** Implement and train TextCNN. Compare accuracy with LSTM.

**Key concept:** TextCNN uses parallel 1D convolutions with different filter sizes to capture n-gram features. Max-over-time pooling picks the most salient feature per filter. Typically reaches 80–85% on movie reviews.

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, filter_sizes, num_classes, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=fs) for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)           # (B, embed, seq)
        pooled = [F.relu(conv(x)).max(dim=2).values       # (B, num_filters)
                  for conv in self.convs]
        x = torch.cat(pooled, dim=1)                      # (B, num_filters * n_sizes)
        return self.fc(self.dropout(x))

def train_model(model, train_loader, test_loader, epochs=5, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.to(device)
    for epoch in range(1, epochs + 1):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        # Evaluate
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for X_b, y_b in test_loader:
                preds.extend(model(X_b.to(device)).argmax(1).cpu().tolist())
                targets.extend(y_b.tolist())
        acc = accuracy_score(targets, preds)
        print(f'  Epoch {epoch}: accuracy={acc:.4f}')
    return acc

print('Training TextCNN...')
cnn_model = TextCNN(len(vocab), embed_dim=64, num_filters=64,
                    filter_sizes=[2, 3, 4], num_classes=2)
cnn_acc = train_model(cnn_model, train_loader, test_loader, epochs=5)
print(f'TextCNN final accuracy: {cnn_acc:.4f}')

---
## Exercise 10.2 — Bidirectional LSTM with Attention

**Task:** Add bidirectional LSTM and attention pooling.

**Key concept:** Attention pooling learns a *weighted* combination of hidden states instead of just taking the last hidden state. The model learns which positions are most informative for classification.

In [ ]:
class BiLSTMWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm       = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout    = nn.Dropout(dropout)
        self.attn_w     = nn.Linear(hidden_dim * 2, 1)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def attention(self, lstm_output):
        # lstm_output: (B, seq, hidden*2)
        scores  = self.attn_w(lstm_output).squeeze(-1)  # (B, seq)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)  # (B, seq, 1)
        return (lstm_output * weights).sum(dim=1)       # (B, hidden*2)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        out, _ = self.lstm(emb)
        context = self.attention(out)
        return self.classifier(self.dropout(context))

print('Training BiLSTM+Attention...')
bilstm_model = BiLSTMWithAttention(len(vocab), embed_dim=64, hidden_dim=64, num_classes=2)
bilstm_acc = train_model(bilstm_model, train_loader, test_loader, epochs=5)
print(f'BiLSTM+Attention final accuracy: {bilstm_acc:.4f}')
print(f'Improvement over TextCNN: {bilstm_acc - cnn_acc:+.4f}')

---
## Exercise 10.3 — Hyperparameter Experiments

**Task:** Compare 3 hyperparameter configurations. Plot val accuracy vs epochs.

**Key concept:** Larger models don't always win — a model that's too large for the dataset overfits. Dropout rate is a key regularizer. Plot training curves to diagnose underfitting vs overfitting.

In [ ]:
def train_with_history(model, train_loader, test_loader, epochs=8, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.to(device)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            criterion(model(X_b), y_b).backward()
            optimizer.step()
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for X_b, y_b in test_loader:
                preds.extend(model(X_b.to(device)).argmax(1).cpu().tolist())
                targets.extend(y_b.tolist())
        history.append(accuracy_score(targets, preds))
    return history

configs = [
    {'label': 'Small  (emb=50, hid=64,  drop=0.3)', 'embed_dim': 50,  'hidden_dim': 64,  'dropout': 0.3},
    {'label': 'Medium (emb=100, hid=128, drop=0.3)', 'embed_dim': 100, 'hidden_dim': 128, 'dropout': 0.3},
    {'label': 'Large  (emb=200, hid=256, drop=0.5)', 'embed_dim': 200, 'hidden_dim': 256, 'dropout': 0.5},
]

plt.figure(figsize=(9, 5))
for cfg in configs:
    m = BiLSTMWithAttention(len(vocab), cfg['embed_dim'], cfg['hidden_dim'], 2, cfg['dropout'])
    hist = train_with_history(m, train_loader, test_loader, epochs=8)
    plt.plot(range(1, len(hist)+1), hist, marker='o', label=cfg['label'])
    print(f"{cfg['label']}: best={max(hist):.4f}")

plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy')
plt.title('Hyperparameter Comparison — BiLSTM Sentiment')
plt.legend(); plt.tight_layout(); plt.show()

---
## Exercise 10.4 — GloVe Pretrained Embeddings

**Task:** Load GloVe and initialize the embedding layer. Compare convergence vs random init.

**Key concept:** Pretrained embeddings provide a warm start — the model begins with linguistic knowledge rather than random noise. This typically leads to faster convergence and better accuracy on small datasets.

In [ ]:
def load_glove_matrix(vocab, embed_dim=50):
    """
    Load GloVe via gensim.downloader and build an embedding matrix
    aligned to our vocab. Words not in GloVe get random vectors.
    """
    import gensim.downloader as api
    print('Downloading glove-wiki-gigaword-50 (~65 MB)...')
    glove = api.load(f'glove-wiki-gigaword-{embed_dim}')

    matrix = np.random.normal(0, 0.1, (len(vocab), embed_dim)).astype(np.float32)
    matrix[0] = 0  # padding stays zero
    found = 0
    for word, idx in vocab.items():
        if word in glove:
            matrix[idx] = glove[word]
            found += 1
    print(f'GloVe coverage: {found}/{len(vocab)} ({100*found/len(vocab):.1f}%)')
    return matrix

try:
    glove_matrix = load_glove_matrix(vocab, embed_dim=50)

    def make_model_glove(vocab, matrix):
        model = BiLSTMWithAttention(len(vocab), embed_dim=50, hidden_dim=64, num_classes=2)
        model.embedding.weight.data.copy_(torch.from_numpy(matrix))
        model.embedding.weight.requires_grad = True  # allow fine-tuning
        return model

    def make_model_random(vocab):
        return BiLSTMWithAttention(len(vocab), embed_dim=50, hidden_dim=64, num_classes=2)

    print('\nTraining with random init...')
    rand_hist = train_with_history(make_model_random(vocab), train_loader, test_loader, epochs=8)
    print('Training with GloVe init...')
    glove_hist = train_with_history(make_model_glove(vocab, glove_matrix), train_loader, test_loader, epochs=8)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, 9), rand_hist,  marker='o', label='Random init')
    plt.plot(range(1, 9), glove_hist, marker='s', label='GloVe init')
    plt.xlabel('Epoch'); plt.ylabel('Val Accuracy')
    plt.title('GloVe vs Random Initialization')
    plt.legend(); plt.tight_layout(); plt.show()
    print(f'Random best: {max(rand_hist):.4f} | GloVe best: {max(glove_hist):.4f}')

except Exception as e:
    print(f'GloVe download failed ({e}). Showing code structure only.')
    print('Expected outcome: GloVe init converges faster (higher accuracy at epoch 1-3).')

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 10.1 | TextCNN: parallel convolutions capture n-gram features; max-pool selects the best |
| 10.2 | Attention pooling: learns a soft importance weight per position |
| 10.3 | Larger ≠ better on small data; dropout is the key regularizer to tune |
| 10.4 | Pretrained GloVe gives faster convergence and ~2–5% accuracy boost |

---
*NLP Course 2027 — Week 05*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**